In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "langchain", "langchain-openai", "langgraph", "duckduckgo-search", "pydantic", "python-dotenv", "langfuse"])
print("✅ All packages installed")

In [ ]:
import json
import sqlite3
import math
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict, Optional, Annotated
import operator

from langfuse.langchain import CallbackHandler as LangfuseCallbackHandler

langfuse_handler = LangfuseCallbackHandler()

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

print("✅ Imports done")
print("✅ Langfuse tracing enabled")

In [48]:
# ── Agent State ──────────────────────────────────────────────────────────────
class InvestmentState(TypedDict):
    # User profile
    name: str
    age: int
    monthly_income: float          # in INR
    investment_amount: float       # in INR
    industry: str

    # Accumulated research
    user_profile_summary: Optional[str]    # risk appetite, goals
    companies: list[str]                   # companies found
    research: list[str]                    # research notes
    summary: str                           # final recommendation

    # Routing
    next_node: Optional[str]               # routing decision
    iteration: int                         # loop counter (max 2 loops)

print("✅ State defined")

✅ State defined


In [ ]:
# ── Tools ────────────────────────────────────────────────────────────────────

# 1. Web Search Tool
@tool
def web_search(query: str) -> str:
    """Search the internet for up-to-date information on companies, markets, or financial news."""
    try:
        from ddgs import DDGS
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)
    except ImportError:
        from duckduckgo_search import DDGS
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)
    except Exception as e:
        return f"Search failed: {e}"


# 2. Calculator Tool
@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression for financial calculations.
    Examples: '50000 * 0.12', '5000 * 12 * 5', 'math.sqrt(144)'
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"


# 3. Coding Tool — run Python for data analysis
@tool
def run_python(code: str) -> str:
    """
    Execute Python code for data analysis (returns the value of the last expression).
    Use for portfolio calculations, percentage returns, risk scoring, etc.
    """
    try:
        namespace = {"math": math, "json": json}
        exec(code, namespace)
        return str(namespace.get("result", "Code executed successfully"))
    except Exception as e:
        return f"Code error: {e}"


# 4. Database Tool — SQLite in-memory store
_db = sqlite3.connect(":memory:")
_db.execute("""
    CREATE TABLE investment_data (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        category TEXT,
        key TEXT,
        value TEXT
    )
""")
_db.commit()

@tool
def database_query(sql: str) -> str:
    """
    Run a SQL query against the investment data store.
    Table: investment_data(id, category, key, value)
    Use INSERT to save research; SELECT to retrieve it.
    """
    try:
        cur = _db.execute(sql)
        _db.commit()
        rows = cur.fetchall()
        return json.dumps(rows) if rows else "Query executed successfully"
    except Exception as e:
        return f"DB error: {e}"


TOOLS = [web_search, calculator, run_python, database_query]
llm_with_tools = llm.bind_tools(TOOLS)

print("✅ Tools defined: web_search, calculator, run_python, database_query")

In [ ]:
# ── Helper: run an agent node with tool loop ──────────────────────────────────
def run_agent_node(system_prompt: str, user_message: str, tools: list) -> str:
    """
    Runs an LLM agent that can call tools in a loop until it produces a final answer.
    Returns the final text response.
    """
    from langchain_core.messages import AIMessage, ToolMessage

    tool_map = {t.name: t for t in tools}
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_message)
    ]

    for _ in range(10):  # max 10 tool calls per node
        response: AIMessage = llm_with_tools.invoke(messages, config={"callbacks": [langfuse_handler]})
        messages.append(response)

        if not response.tool_calls:
            return response.content  # final answer

        # Execute each tool call
        for tc in response.tool_calls:
            tool_fn = tool_map.get(tc["name"])
            if tool_fn:
                result = tool_fn.invoke(tc["args"])
                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return messages[-1].content if hasattr(messages[-1], "content") else "Agent finished"

print("✅ Agent runner helper defined (with Langfuse tracing)")

In [51]:
# ── Node 1: User Research ─────────────────────────────────────────────────────
def user_research(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("🔍 NODE: User Research")
    print(f"{'='*50}")

    system_prompt = """
You are a certified financial planner. Your job is to analyse a client's profile
and determine:
1. Risk appetite (conservative / moderate / aggressive) based on age and income
2. Investment goals (growth, income, capital preservation)
3. Suitable investment horizon
4. Recommended investment allocation strategy for the given amount

Use the calculator tool to compute key ratios (e.g. investment-to-income ratio,
estimated returns at different rates).
Save your findings to the database using the database_query tool.
"""

    user_message = f"""
Client profile:
- Name: {state['name']}
- Age: {state['age']}
- Monthly income: ₹{state['monthly_income']:,.0f}
- Investment amount: ₹{state['investment_amount']:,.0f}
- Target industry: {state['industry']}

Analyse this profile and produce a concise investment profile summary.
"""

    profile_summary = run_agent_node(system_prompt, user_message, TOOLS)
    print(f"\n{profile_summary}")

    return {
        "user_profile_summary": profile_summary,
        "next_node": "market_research",
        "iteration": state.get("iteration", 0)
    }


# ── Node 2: Market Research ───────────────────────────────────────────────────
def market_research(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("📊 NODE: Market Research")
    print(f"{'='*50}")

    system_prompt = """
You are a market research analyst specialising in equity and sector investments.
Your job is to:
1. Research current trends and outlook for the target industry
2. Identify top-performing segments within that industry
3. Assess market risks (inflation, regulation, competition)
4. Suggest 3-5 specific publicly listed companies worth researching further

Use web_search to get current data. Save company names to the database.
At the end, decide if you need more user profile clarification (next: user_research)
or can proceed to company research (next: company_research).
Output your next step as: NEXT_NODE: <node_name>
"""

    user_message = f"""
User profile:
{state['user_profile_summary']}

Target industry: {state['industry']}
Investment amount: ₹{state['investment_amount']:,.0f}
Iteration: {state.get('iteration', 0)}

Research the {state['industry']} sector and recommend companies to analyse.
"""

    research_result = run_agent_node(system_prompt, user_message, TOOLS)
    print(f"\n{research_result}")

    # Extract next node from LLM output
    next_node = "company_research"
    if "NEXT_NODE: user_research" in research_result:
        next_node = "user_research"

    # Extract company names (lines that look like company names)
    companies = state.get("companies", [])
    research = state.get("research", [])
    research.append(f"[Market Research]\n{research_result}")

    return {
        "research": research,
        "companies": companies,
        "next_node": next_node,
        "iteration": state.get("iteration", 0) + 1
    }


# ── Node 3: Company Research ──────────────────────────────────────────────────
def company_research(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("🏢 NODE: Company Research")
    print(f"{'='*50}")

    system_prompt = """
You are an equity research analyst. For each company suggested:
1. Search for recent financials (revenue growth, P/E ratio, market cap)
2. Check recent news and sentiment
3. Calculate potential returns using the calculator tool
4. Rate each company: BUY / HOLD / AVOID with a reason
5. Suggest how to allocate the investment amount across top picks

Use web_search for current data and calculator for return projections.
At the end, decide if you need more market data (next: market_research)
or are ready to summarise (next: summarise).
Output your next step as: NEXT_NODE: <node_name>
"""

    market_notes = "\n".join(state.get("research", [])[-2:])  # last 2 notes

    user_message = f"""
User profile:
{state['user_profile_summary']}

Market research notes:
{market_notes}

Investment amount: ₹{state['investment_amount']:,.0f}
Industry: {state['industry']}

Research the top companies and provide BUY/HOLD/AVOID ratings with allocation suggestions.
"""

    company_result = run_agent_node(system_prompt, user_message, TOOLS)
    print(f"\n{company_result}")

    # Extract next node
    next_node = "summarise"
    if "NEXT_NODE: market_research" in company_result:
        next_node = "market_research"

    research = state.get("research", [])
    research.append(f"[Company Research]\n{company_result}")

    return {
        "research": research,
        "next_node": next_node,
        "iteration": state.get("iteration", 0) + 1
    }


# ── Node 4: Summarise ─────────────────────────────────────────────────────────
def summarise(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("📋 NODE: Summarise")
    print(f"{'='*50}")

    system_prompt = """
You are a senior investment advisor. Compile all research into a clear,
structured investment recommendation report. Include:
1. Client profile snapshot
2. Market outlook
3. Top investment picks with ratings
4. Suggested allocation table
5. Risk warnings
6. Expected returns (conservative / moderate / optimistic)

Use the calculator tool to compute final return projections.
Format the report clearly with sections and bullet points.
"""

    all_research = "\n\n".join(state.get("research", []))

    user_message = f"""
Client: {state['name']}, Age: {state['age']}
Investment: ₹{state['investment_amount']:,.0f} in {state['industry']}

User Profile Analysis:
{state['user_profile_summary']}

All Research Notes:
{all_research}

Produce the final investment recommendation report.
"""

    final_report = run_agent_node(system_prompt, user_message, TOOLS)

    return {"summary": final_report}


print("✅ All 4 agent nodes defined")

✅ All 4 agent nodes defined


In [52]:
# ── Routing Logic ─────────────────────────────────────────────────────────────
def route_market_research(state: InvestmentState) -> str:
    """After market research: loop back to user_research or proceed to company_research."""
    if state.get("next_node") == "user_research" and state.get("iteration", 0) < 2:
        print("↩️  Routing back to User Research for clarification")
        return "user_research"
    print("➡️  Routing to Company Research")
    return "company_research"


def route_company_research(state: InvestmentState) -> str:
    """After company research: loop back to market_research or proceed to summarise."""
    if state.get("next_node") == "market_research" and state.get("iteration", 0) < 3:
        print("↩️  Routing back to Market Research for more data")
        return "market_research"
    print("➡️  Routing to Summarise")
    return "summarise"


print("✅ Routing functions defined")

✅ Routing functions defined


In [53]:
# ── Build the LangGraph ───────────────────────────────────────────────────────
graph = StateGraph(InvestmentState)

# Add nodes
graph.add_node("user_research", user_research)
graph.add_node("market_research", market_research)
graph.add_node("company_research", company_research)
graph.add_node("summarise", summarise)

# Entry point
graph.add_edge(START, "user_research")

# Fixed edge: user_research → market_research
graph.add_edge("user_research", "market_research")

# Conditional edge: market_research → (user_research | company_research)
graph.add_conditional_edges(
    "market_research",
    route_market_research,
    {"user_research": "user_research", "company_research": "company_research"}
)

# Conditional edge: company_research → (market_research | summarise)
graph.add_conditional_edges(
    "company_research",
    route_company_research,
    {"market_research": "market_research", "summarise": "summarise"}
)

# Fixed edge: summarise → END
graph.add_edge("summarise", END)

# Compile
investment_agent = graph.compile()

print("✅ Investment Agent graph compiled!")
print("Flow: User Research → Market Research → Company Research → Summarise")
print("      (with loop-backs based on agent decisions)")

✅ Investment Agent graph compiled!
Flow: User Research → Market Research → Company Research → Summarise
      (with loop-backs based on agent decisions)


In [ ]:
# ── Run the Agent ─────────────────────────────────────────────────────────────
from langfuse import get_client

initial_state: InvestmentState = {
    "name": "Praveen",
    "age": 60,
    "monthly_income": 500000,      # ₹5 Lakhs/month
    "investment_amount": 50000,    # ₹50,000
    "industry": "Food Industry",
    "user_profile_summary": None,
    "companies": [],
    "research": [],
    "summary": "",
    "next_node": None,
    "iteration": 0
}

print("🚀 Starting Investment Agent for Praveen...")
print(f"   Age: {initial_state['age']} | Income: ₹{initial_state['monthly_income']:,.0f}/month")
print(f"   Investment: ₹{initial_state['investment_amount']:,.0f} | Industry: {initial_state['industry']}")
print()

result = investment_agent.invoke(initial_state, config={"callbacks": [langfuse_handler]})

get_client().flush()
print("\n✅ Langfuse trace flushed — check your Langfuse dashboard")

In [ ]:
# ── Final Investment Report ───────────────────────────────────────────────────
print("\n" + "="*60)
print("       INVESTMENT RECOMMENDATION REPORT")
print("="*60)
print(result["summary"])
print("="*60)